In [1]:
pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-7sbk766l
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-7sbk766l
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36590 sha256=2e0b2ee4d12e467b7032190ff91b6fba69357a8f70fbb88e12feb94836a71750
  Stored in directory: /tmp/pip-ephem-wheel-cache-znz13s3z/wheels/29/82/ff/04e2be9805a1cb48bec0b85b5a6da6b63f647645750a0e42d4
Successfully built segment_anything

[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from skimage import measure
from matplotlib.patches import Polygon

# Dataset class (already provided)
class MVTecLOCODataset(Dataset):
    def __init__(self, root_dir, split='train', anomaly_type='logical_anomalies', transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []  # 0 for normal, 1 for anomaly

        if split == 'train':
            # Load 'good' images for training
            good_dir = os.path.join(root_dir, 'train', 'good')
            self._load_images_from_folder(good_dir, label=0)
        elif split == 'test':
            # Load 'good' and specified anomalies for testing
            good_dir = os.path.join(root_dir, 'test', 'good')
            anomaly_dir = os.path.join(root_dir, 'test', anomaly_type)
            self._load_images_from_folder(good_dir, label=0)
            self._load_images_from_folder(anomaly_dir, label=1)
        else:
            raise ValueError(f"Invalid split: {split}")

    def _load_images_from_folder(self, folder, label):
        if not os.path.isdir(folder):
            print(f"Folder {folder} is missing!")
            return
        for root, _, files in os.walk(folder):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(root, file)
                    self.image_paths.append(img_path)
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = self.labels[idx]
        return image, label, img_path

In [2]:
# Autoencoder model
class Autoencoder(nn.Module):
    def __init__(self, embedding_size=(256, 64, 64)):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, stride=2, padding=1),  # (128, 32, 32)
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, stride=2, padding=1),   # (64, 16, 16)
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, stride=2, padding=1),    # (32, 8, 8)
            nn.ReLU(),
            nn.Conv2d(32, 16, kernel_size=3, stride=2, padding=1),    # (16, 4, 4)
            nn.ReLU(),
            nn.Flatten(),                                             # (16*4*4)
            nn.Linear(16*4*4, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 16*4*4),
            nn.ReLU(),
            nn.Unflatten(1, (16, 4, 4)),
            nn.ConvTranspose2d(16, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # (32, 8, 8)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 64, kernel_size=3, stride=2, padding=1, output_padding=1),  # (64, 16, 16)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 128, kernel_size=3, stride=2, padding=1, output_padding=1), # (128, 32, 32)
            nn.ReLU(),
            nn.ConvTranspose2d(128, 256, kernel_size=3, stride=2, padding=1, output_padding=1),# (256, 64, 64)
            nn.Sigmoid(),  # Output between 0 and 1
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [3]:
# Evaluation method
def evaluate_model(autoencoder, image_encoder, test_loader, device):
    autoencoder.eval()
    image_encoder.eval()
    
    all_labels = []
    all_scores = []
    
    with torch.no_grad():
        for images, labels, _ in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            embeddings = image_encoder(images)
            outputs = autoencoder(embeddings)
            
            loss = torch.mean((outputs - embeddings) ** 2, dim=[1, 2, 3])  # MSE per sample
            
            all_labels.extend(labels.cpu().numpy())
            all_scores.extend(loss.cpu().numpy())
    
    # Compute AUROC
    auroc = roc_auc_score(all_labels, all_scores)
    print(f"AUROC: {auroc:.4f}")

# Visualization method
def visualize_masks(image, masks):
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    ax = plt.gca()
    for mask in masks:
        color = np.random.rand(3,)
        m = mask['segmentation']
        contours = measure.find_contours(m, 0.5)
        for contour in contours:
            contour = np.flip(contour, axis=1)
            polygon = Polygon(contour, facecolor=color, edgecolor=color, alpha=0.5)
            ax.add_patch(polygon)
    plt.axis('off')
    plt.show()

def visualize_image_and_masks(image_path, sam_model):
    image = Image.open(image_path).convert('RGB')
    image_np = np.array(image)
    
    mask_generator = SamAutomaticMaskGenerator(sam_model)
    masks = mask_generator.generate(image_np)
    
    visualize_masks(image_np, masks)

In [4]:
# Main function
def main():
    # Device configuration
    device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

    # Data transformations
    sam_transform = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],  # ImageNet mean
            std=[0.229, 0.224, 0.225]     # ImageNet std
        )
    ])

    # Paths
    root_dir = "/home/hafiz/my_thesis/seg_recons/dataset/screw_bag"  # Update with your dataset path
    model_type = "vit_b"
    sam_checkpoint = "sam_vit_b_01ec64.pth"  # Update with your SAM checkpoint path

    # Load datasets
    train_dataset = MVTecLOCODataset(root_dir=root_dir, split='train', transform=sam_transform)
    test_dataset = MVTecLOCODataset(root_dir=root_dir, split='test', transform=sam_transform)
    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=4)

    # Load SAM model
    sam = sam_model_registry[model_type](checkpoint=sam_checkpoint).to(device)
    sam.eval()
    image_encoder = sam.image_encoder

    # Initialize autoencoder
    autoencoder = Autoencoder().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(autoencoder.parameters(), lr=1e-4)

    # Training loop
    num_epochs = 20
    save_interval = 5
    for epoch in range(num_epochs):
        autoencoder.train()
        running_loss = 0.0
        for images, _, _ in train_loader:
            images = images.to(device)
            
            with torch.no_grad():
                embeddings = image_encoder(images)
            
            outputs = autoencoder(embeddings)
            loss = criterion(outputs, embeddings)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")
        
        # Save checkpoint
        if (epoch + 1) % save_interval == 0:
            checkpoint_path = f'checkpoint_epoch_{epoch+1}.pth'
            torch.save({
                'epoch': epoch + 1,
                'autoencoder_state_dict': autoencoder.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': epoch_loss,
            }, checkpoint_path)
            print(f"Checkpoint saved at {checkpoint_path}")

    # Evaluation
    evaluate_model(autoencoder, image_encoder, test_loader, device)

    # Visualization (example with one test image)
    sample_image_path = test_dataset.image_paths[0]  # Replace with desired image path
    visualize_image_and_masks(sample_image_path, sam)

if __name__ == "__main__":
    main()


/home/hafiz/my_thesis/seg_recons/venv/lib/python3.12/site-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f

RuntimeError: DataLoader worker (pid(s) 514192) exited unexpectedly